In [ ]:
import json
import os
import sys
from typing import Dict, Any
import csv

# Target data file
DATABASE_FILE = "sample_records.json"

# In-memory storage: Key = Student ID, Value = Dict of Student attributes
STUDENT_REGISTRY: Dict[str, Dict[str, Any]] = {}


def load_records_from_json(file_path: str) -> Dict[str, Dict[str, Any]]:
    """Loads student records safely from a JSON file.

    Handles FileNotFoundError and corrupted JSON formatting gracefully.
    """
    if not os.path.exists(file_path):
        print(f"[WARN] Database file '{file_path}' not found. Starting with empty registry.")
        return {}

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
            print(f"[SUCCESS] Loaded {len(data)} record(s) from {file_path}.")
            return data
    except json.JSONDecodeError as json_err:
        print(f"[ERROR] Corrupted JSON structure in '{file_path}': {json_err}")
        return {}
    except Exception as err:
        print(f"[UNEXPECTED ERROR] Failed to load data: {err}")
        return {}


def view_all_records(registry: Dict[str, Dict[str, Any]]) -> None:
    """Prints all student records in a formatted tabular view."""
    if not registry:
        print("\n[INFO] No records found in the registry.")
        return

    separator = "-" * 75
    print("\n" + separator)
    print(f"{'Student ID':<12} | {'Name':<22} | {'Branch':<22} | {'CGPA':<5}")
    print(separator)
    for student_id, details in registry.items():
        name = details.get("name", "N/A")
        branch = details.get("branch", "N/A")
        cgpa = details.get("cgpa", 0.0)
        print(f"{student_id:<12} | {name:<22} | {branch:<22} | {cgpa:<5.2f}")
    print(separator + "\n")


# ==============================================================================
# ✍️ STUDENT TASKS TO IMPLEMENT BELOW (TODO SECTION)
# ==============================================================================

def add_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 1: Prompt user for student ID, name, branch, and CGPA.

    Validation Rules:
      1. Student ID must not already exist in the registry.
      2. Name and Branch must not be empty after stripping whitespace.
      3. CGPA must be a valid float between 0.0 and 10.0.
      4. Auto-generate email: <first_name_lowercase>.<id_lowercase>@university.edu
    """
    print("\n--- Add New Student ---")
    # TODO: Implement prompt, validation, and addition to registry dictionary
    sid, name, branch = input("ID: ").strip(), input("Name: ").strip(), input("Branch: ").strip()
    if not sid or sid in registry or not name or not branch:
        return print("[ERROR] Invalid ID, Name, or Branch.")
    try:
        cgpa = float(input("CGPA (0-10): ").strip())
        if not (0 <= cgpa <= 10): raise ValueError
    except ValueError:
        return print("[ERROR] CGPA must be a float between 0 and 10.")
    registry[sid] = {"name": name, "branch": branch, "cgpa": cgpa, "email": f"{name.split()[0].lower()}.{sid.lower()}@university.edu"}
    print(f"[SUCCESS] Added '{name}'.")


def search_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 2: Search by student ID (exact) or student Name (case-insensitive substring)."""
    print("\n--- Search Student Records ---")
    # TODO: Implement search logic
    q = input("Search ID/Name: ").strip().lower()
    res = {k: v for k, v in registry.items() if q in (k.lower(), v.get("name", "").lower())}
    if res:
        view_all_records(res)
    else:
        print("[INFO] No records found.")


def delete_student_record(registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 3: Prompt for Student ID and delete record with confirmation."""
    print("\n--- Delete Student Record ---")
    # TODO: Implement delete logic with confirmation
    sid = input("ID to delete: ").strip()
    if sid not in registry: return print("[ERROR] ID not found.")
    if input(f"Delete '{registry[sid]['name']}' (y/N)? ").strip().lower() == 'y':
        del registry[sid]
        print("[SUCCESS] Deleted.")


def save_records_to_json(file_path: str, registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 4: Serialize the in-memory registry dictionary to the JSON file safely."""
    # TODO: Open file in write mode and use json.dump(..., indent=2)
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(registry, file, indent=2)
    print(f"[SUCCESS] Saved {len(registry)} records.")


def export_to_csv(file_path: str, registry: Dict[str, Dict[str, Any]]) -> None:
    """TODO Task 5 (Bonus): Export all student records to a CSV file."""
    # TODO: Use the csv module (DictWriter) to write records
    if not registry: return print("[INFO] Registry empty.")
    with open(file_path, "w", newline="", encoding="utf-8") as file:
        w = csv.DictWriter(file, fieldnames=["id", "name", "branch", "cgpa", "email"])
        w.writeheader()
        w.writerows([{"id": k, **v} for k, v in registry.items()])
    print("[SUCCESS] Exported to CSV.")



def main_menu() -> None:
    """Main CLI control loop."""
    global STUDENT_REGISTRY
    STUDENT_REGISTRY = load_records_from_json(DATABASE_FILE)

    menu_banner = """
========================================
🎓 STUDENT RECORD MANAGEMENT SYSTEM
========================================
1. View All Records
2. Add Student Record
3. Search Record
4. Delete Record
5. Save Database to File
6. Export Records to CSV (Bonus)
0. Save & Exit
========================================
"""
    while True:
        print(menu_banner)
        choice = input("Enter choice [0-6]: ").strip()

        if choice == "1":
            view_all_records(STUDENT_REGISTRY)
        elif choice == "2":
            add_student_record(STUDENT_REGISTRY)
        elif choice == "3":
            search_student_record(STUDENT_REGISTRY)
        elif choice == "4":
            delete_student_record(STUDENT_REGISTRY)
        elif choice == "5":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
        elif choice == "6":
            export_to_csv("students_export.csv", STUDENT_REGISTRY)
        elif choice == "0":
            save_records_to_json(DATABASE_FILE, STUDENT_REGISTRY)
            print("[INFO] Application closed successfully. Good bye!")
            sys.exit(0)
        else:
            print("[WARN] Invalid option selected. Please enter a number between 0 and 6.")


if __name__ == "__main__":
    main_menu()